# Step 3: Analyze Jailbreak Features

This notebook identifies and analyzes SAE features related to jailbreak attempts.

In [ ]:
import sys
sys.path.append('..')

import torch
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.models import load_gemma_models, BatchTopKSAE
from src.analysis import JailbreakAnalyzer

## Load Configuration and Models

In [ ]:
with open('../configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

In [ ]:
# Load models
print("Loading Gemma 2 models...")
models = load_gemma_models('../configs/config.yaml')
print(models)

In [ ]:
# Load trained SAE
checkpoint_path = '../checkpoints/final_model.pt'

print(f"Loading SAE from {checkpoint_path}...")
checkpoint = torch.load(checkpoint_path, map_location=device)

sae = BatchTopKSAE.from_config(checkpoint['config'])
sae.load_state_dict(checkpoint['model_state_dict'])
sae = sae.to(device)
sae.eval()

print("SAE loaded successfully!")

## Initialize Jailbreak Analyzer

In [ ]:
analyzer = JailbreakAnalyzer(
    models,
    sae,
    config['models']['target_layer'],
    device
)

print("Analyzer initialized!")

## Load Prompts

In [ ]:
jailbreak_prompts = analyzer.load_jailbreak_prompts('../data/prompts/jailbreak_prompts.json')
safe_prompts = analyzer.load_safe_prompts()

print(f"Loaded {len(jailbreak_prompts)} jailbreak prompts")
print(f"Loaded {len(safe_prompts)} safe prompts")

print("\nExample jailbreak prompt:")
print(f"  {jailbreak_prompts[0]}")

print("\nExample safe prompt:")
print(f"  {safe_prompts[0]}")

## Identify Jailbreak Features

In [ ]:
print("Identifying jailbreak features...")
print("This may take a few minutes...\n")

features_df = analyzer.identify_jailbreak_features(
    jailbreak_prompts=jailbreak_prompts,
    safe_prompts=safe_prompts,
    top_k=50
)

print("\nTop 10 jailbreak features:")
features_df.head(10)

## Visualize Feature Statistics

In [ ]:
# Plot top features
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Differential activation
top_10 = features_df.head(10)
axes[0, 0].barh(range(len(top_10)), top_10['diff'])
axes[0, 0].set_yticks(range(len(top_10)))
axes[0, 0].set_yticklabels(top_10['latent_idx'])
axes[0, 0].set_xlabel('Differential Activation')
axes[0, 0].set_ylabel('Latent Index')
axes[0, 0].set_title('Top 10 Features by Differential Activation')

# 2. Effect size (Cohen's d)
axes[0, 1].barh(range(len(top_10)), top_10['cohens_d'])
axes[0, 1].set_yticks(range(len(top_10)))
axes[0, 1].set_yticklabels(top_10['latent_idx'])
axes[0, 1].set_xlabel("Cohen's d (Effect Size)")
axes[0, 1].set_ylabel('Latent Index')
axes[0, 1].set_title('Top 10 Features by Effect Size')

# 3. Jailbreak vs Safe activations
x = range(len(top_10))
width = 0.35
axes[1, 0].bar([i - width/2 for i in x], top_10['jailbreak_mean'], width, label='Jailbreak', alpha=0.8)
axes[1, 0].bar([i + width/2 for i in x], top_10['safe_mean'], width, label='Safe', alpha=0.8)
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(top_10['latent_idx'], rotation=45)
axes[1, 0].set_ylabel('Mean Activation')
axes[1, 0].set_xlabel('Latent Index')
axes[1, 0].set_title('Jailbreak vs Safe Mean Activations')
axes[1, 0].legend()

# 4. Scatter plot
axes[1, 1].scatter(features_df['jailbreak_mean'], features_df['safe_mean'], alpha=0.5)
axes[1, 1].plot([0, features_df[['jailbreak_mean', 'safe_mean']].max().max()],
                [0, features_df[['jailbreak_mean', 'safe_mean']].max().max()],
                'r--', alpha=0.5)
axes[1, 1].set_xlabel('Jailbreak Mean Activation')
axes[1, 1].set_ylabel('Safe Mean Activation')
axes[1, 1].set_title('Jailbreak vs Safe Activations (All Features)')

plt.tight_layout()
plt.show()

## Examine Top Jailbreak Feature

In [ ]:
# Get the top jailbreak feature
top_feature_idx = features_df.iloc[0]['latent_idx']

print(f"Analyzing top feature: {int(top_feature_idx)}\n")

# Find top activating jailbreak examples
print("Top activating jailbreak prompts:")
top_jailbreak = analyzer.find_top_activating_examples(
    int(top_feature_idx),
    jailbreak_prompts,
    top_k=5
)

for i, (prompt, activation) in enumerate(top_jailbreak, 1):
    print(f"{i}. [{activation:.4f}] {prompt}")

print("\nTop activating safe prompts:")
top_safe = analyzer.find_top_activating_examples(
    int(top_feature_idx),
    safe_prompts,
    top_k=5
)

for i, (prompt, activation) in enumerate(top_safe, 1):
    print(f"{i}. [{activation:.4f}] {prompt}")

## Test Feature Steering

In [ ]:
# Test steering with the top feature
test_prompt = jailbreak_prompts[0]

print(f"Test prompt: {test_prompt}\n")
print("Testing steering with different coefficients...\n")

steering_results = analyzer.test_feature_steering(
    test_prompt,
    int(top_feature_idx),
    coefficients=[0.0, 0.5, 1.0, 2.0]
)

for _, row in steering_results.iterrows():
    print(f"Coefficient: {row['coefficient']}")
    print(f"Generated: {row['generated_text'][:200]}...\n")

## Save Results

In [ ]:
# Save jailbreak features
output_path = '../data/jailbreak_features.csv'
analyzer.save_jailbreak_features(features_df, output_path)

print(f"Jailbreak features saved to {output_path}")

## Summary

This notebook identified SAE features that:
1. Activate more strongly on jailbreak prompts vs safe prompts
2. Show significant effect sizes (Cohen's d)
3. Can be used for steering model behavior

Next steps:
- Analyze these features in the KL dashboard
- Test steering on more diverse prompts
- Investigate feature combinations